In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/spotify.db")

In [2]:
pd.read_sql("""
    SELECT country,
           COUNT(DISTINCT spotify_id) AS farkli_sarki
    FROM songs
    GROUP BY country
""", conn)

,country,farkli_sarki
0,GLOBAL,896
1,TR,524


In [3]:
sonuc = pd.read_sql("""
    SELECT COUNT(DISTINCT spotify_id) AS sadece_tr
    FROM songs
    WHERE country = 'TR'
      AND spotify_id NOT IN (
          SELECT spotify_id FROM songs WHERE country = 'GLOBAL'
      )
""", conn)

sayi = sonuc.loc[0, "sadece_tr"]
print(f"Türkiye'ye özgü şarkı: {sayi} / 524 (%{sayi / 524 * 100:.1f})")

Türkiye'ye özgü şarkı: 514 / 524 (%98.1)


In [4]:
pd.read_sql("""
    SELECT DISTINCT name, artists
    FROM songs
    WHERE country = 'TR'
      AND spotify_id IN (
          SELECT spotify_id FROM songs WHERE country = 'GLOBAL'
      )
""", conn)

,name,artists
0,Die With A Smile,"Lady Gaga, Bruno Mars"
1,São Paulo (feat. Anitta),"The Weeknd, Anitta"
2,APT.,"ROSÉ, Bruno Mars"
3,Not Like Us,Kendrick Lamar
4,São Paulo,The Weeknd
5,Ma Meilleure Ennemie (from the series Arcane L...,"Stromae, Pomme, Arcane"
6,Lan,Zeynep Bastık
7,i like the way you kiss me,Artemas
8,Last Christmas - Single Version,Wham!


In [7]:
pd.read_sql("""
    SELECT country,
           ROUND(AVG(gun_sayisi), 1) AS ortalama_gun
    FROM (
        SELECT country, spotify_id,
               COUNT(DISTINCT snapshot_date) AS gun_sayisi
        FROM songs
        GROUP BY country, spotify_id
    )
    GROUP BY country
""", conn)

,country,ortalama_gun
0,GLOBAL,32.3
1,TR,55.6


In [8]:
pd.read_sql("""
    SELECT country,
           ROUND(AVG(danceability), 3) AS dans,
           ROUND(AVG(energy), 3) AS enerji,
           ROUND(AVG(acousticness), 3) AS akustik,
           ROUND(AVG(valence), 3) AS nese
    FROM (
        SELECT DISTINCT country, spotify_id,
               danceability, energy, acousticness, valence
        FROM songs
    )
    GROUP BY country
""", conn)

,country,dans,enerji,akustik,nese
0,GLOBAL,0.638,0.632,0.270,0.501
1,TR,0.656,0.668,0.272,0.542


In [9]:
pd.read_sql("""
    SELECT name, artists,
           COUNT(DISTINCT snapshot_date) AS gun_sayisi,
           MIN(daily_rank) AS en_iyi_sira
    FROM songs
    WHERE country = 'TR'
    GROUP BY spotify_id
    ORDER BY gun_sayisi DESC
    LIMIT 10
""", conn)

,name,artists,gun_sayisi,en_iyi_sira
0,aklına ben gelicem,BLOK3,519,2
1,Sen Bilmezsin,Dedublüman,408,5
2,Yakışıklı,"KÖFN, Simge",331,4
3,Canın Sağ Olsun,"Semicenk, Rast",328,8
4,Kehribar,"Burak Bulut, Ebru Yaşar",320,3
5,BAYBAY,BLOK3,290,20
6,Yansıma,"Derya Uluğ, Asil Gök",280,1
7,Bertaraf,"Canbay & Wolker, Heijan, Muti",279,4
8,Yüreğim Ağlar,Amo988,276,2
9,Önümüz Yaz,Simge,264,2
